# Clase 12 – Práctica: IA responsable, ética, sesgo e interpretabilidad

Este cuaderno es la práctica autónoma de la Clase 12 del curso Advanced Deep Learning ("aprendizaje profundo avanzado"). Aquí aplicas, con código, las ideas centrales sobre inteligencia artificial (IA) responsable que trabajaste en la clase en vivo: la equidad de un modelo frente a distintos grupos demográficos y la interpretabilidad de sus decisiones.

## Qué vas a practicar

- Auditar un modelo de concesión de crédito ficticio y medir si trata igual a dos grupos demográficos, con la tasa de verdaderos positivos (TPR)
- Construir una técnica sencilla de interpretabilidad que estima qué variable pesa más en las decisiones del modelo

## Requisito previo

Haber visto la Clase 12 ("IA responsable: ética, sesgo e interpretabilidad") y tener claros los conceptos de sesgo algorítmico e interpretabilidad. Si no la has visto todavía, revisa antes el capítulo 13 del manual.

**Ejecuta cada celda de código en orden: los dos ejercicios comparten variables entre sí.**

## Ejercicio 1: audita un modelo de concesión de crédito ficticio

Imagina que trabajas con un banco que usa un modelo de aprendizaje automático para decidir si concede un préstamo. El equipo de datos te entrega un conjunto de datos sintético con dos grupos demográficos, A y B, y un modelo de regresión logística ya entrenado (`LogisticRegression` de scikit-learn) que predice si cada persona solicitante paga el préstamo (etiqueta 1) o no lo paga (etiqueta 0).

El andamiaje de abajo genera los datos —con una diferencia de representación deliberada entre los dos grupos, para que tengas algo real que auditar—, entrena el modelo y te deja el resultado en `X_test`, `y_test`, `grupo_test` y `y_pred`.

Completa la función `calcular_diferencia_equidad(y_real, y_predicho, grupo)`, que audita el modelo con la métrica de diferencia de oportunidad igual (equal opportunity difference): calcula la TPR —la proporción de personas que sí pagan el préstamo (etiqueta real 1) y a las que el modelo también predice 1— por separado en cada grupo, y devuelve la diferencia entre el grupo con mejor TPR y el grupo con peor TPR. Cuanto más cerca de cero, más equitativo es el modelo entre los grupos.

**Pistas:**

- Recorre los grupos únicos presentes en `grupo` con `np.unique(grupo)`
- Para cada grupo, usa una máscara booleana (`grupo == g`) para quedarte solo con las filas de ese grupo
- Dentro de cada grupo, la TPR es la proporción de aciertos del modelo (`y_predicho == 1`) entre las personas cuya etiqueta real es 1
- La diferencia final es el TPR máximo menos el TPR mínimo entre los grupos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42)
n_por_grupo = 1000

# Grupo A: grupo mayoritario, bien representado en el histórico
edad_a = rng.normal(40, 10, n_por_grupo)
ingresos_a = rng.normal(50, 15, n_por_grupo)
antiguedad_a = rng.normal(8, 4, n_por_grupo)
score_a = 0.04 * ingresos_a + 0.3 * antiguedad_a - 5
etiqueta_a = (score_a + rng.normal(0, 3, n_por_grupo) > 0).astype(int)

# Grupo B: grupo minoritario, peor representado en el histórico.
# El proceso causal es el mismo, pero los ingresos declarados son
# sistemáticamente más bajos y la etiqueta tiene más ruido de medición:
# así se simula un sesgo de representación en el origen de los datos.
edad_b = rng.normal(38, 10, n_por_grupo)
ingresos_b = rng.normal(38, 15, n_por_grupo)
antiguedad_b = rng.normal(6, 4, n_por_grupo)
score_b = 0.04 * ingresos_b + 0.3 * antiguedad_b - 5
etiqueta_b = (score_b + rng.normal(0, 5, n_por_grupo) > 0).astype(int)

df = pd.concat([
    pd.DataFrame({"edad": edad_a, "ingresos": ingresos_a, "antiguedad": antiguedad_a,
                  "grupo": "A", "etiqueta": etiqueta_a}),
    pd.DataFrame({"edad": edad_b, "ingresos": ingresos_b, "antiguedad": antiguedad_b,
                  "grupo": "B", "etiqueta": etiqueta_b}),
], ignore_index=True)

X = df[["edad", "ingresos", "antiguedad"]].values
y = df["etiqueta"].values
grupo = df["grupo"].values

X_train, X_test, y_train, y_test, grupo_train, grupo_test = train_test_split(
    X, y, grupo, test_size=0.3, random_state=42, stratify=grupo
)

# Modelo ya entrenado que vas a auditar
modelo = LogisticRegression()
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

nombres_columnas = ["edad", "ingresos", "antiguedad"]

print("Filas de test:", len(X_test))
print("Grupos presentes:", np.unique(grupo_test))

In [ ]:
def calcular_diferencia_equidad(y_real, y_predicho, grupo):
    """
    Calcula la tasa de verdaderos positivos (TPR) de cada grupo demográfico
    y la diferencia entre el grupo con mejor y peor TPR (diferencia de
    oportunidad igual). Un valor cercano a cero indica que el modelo trata
    a ambos grupos con la misma eficacia a la hora de detectar verdaderos
    positivos.

    Parámetros
    ----------
    y_real : array con la etiqueta real (0/1)
    y_predicho : array con la etiqueta predicha por el modelo (0/1)
    grupo : array con el grupo demográfico de cada fila (por ejemplo, "A" o "B")

    Devuelve
    --------
    tprs : dict {grupo: TPR}
    diferencia : diferencia entre el TPR máximo y el TPR mínimo
    """
    grupos_unicos = np.unique(grupo)
    tprs = {}

    for g in grupos_unicos:
        mascara_grupo = grupo == g
        y_real_grupo = y_real[mascara_grupo]
        y_predicho_grupo = y_predicho[mascara_grupo]

        # TODO: calcula la tasa de verdaderos positivos (TPR) de este grupo:
        # de las filas donde y_real_grupo es igual a 1, qué proporción tiene
        # también y_predicho_grupo igual a 1
        tpr_grupo = ...

        tprs[g] = tpr_grupo

    # TODO: calcula la diferencia entre el TPR máximo y el TPR mínimo de tprs
    diferencia = ...

    return tprs, diferencia


tprs_por_grupo, diferencia = calcular_diferencia_equidad(y_test, y_pred, grupo_test)

print("TPR por grupo:")
for g, tpr in tprs_por_grupo.items():
    print(f"  Grupo {g}: {tpr:.3f}")
print(f"\nDiferencia de oportunidad igual: {diferencia:.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(list(tprs_por_grupo.keys()), list(tprs_por_grupo.values()),
       color=["#4C72B0", "#DD8452"], width=0.5)
ax.set_ylim(0, 1)
ax.set_ylabel("Tasa de verdaderos positivos (TPR)")
ax.set_title("Auditoría de equidad: TPR por grupo demográfico")
plt.tight_layout()
plt.show()

<details>
<summary>💡 Ver solución</summary>

```python
def calcular_diferencia_equidad(y_real, y_predicho, grupo):
    """
    Calcula la tasa de verdaderos positivos (TPR) de cada grupo demográfico
    y la diferencia entre el grupo con mejor y peor TPR (diferencia de
    oportunidad igual).
    """
    grupos_unicos = np.unique(grupo)
    tprs = {}

    for g in grupos_unicos:
        mascara_grupo = grupo == g
        y_real_grupo = y_real[mascara_grupo]
        y_predicho_grupo = y_predicho[mascara_grupo]

        # TPR = proporción de positivos reales que el modelo también predice como positivos
        positivos_reales = y_real_grupo == 1
        if positivos_reales.sum() == 0:
            tpr_grupo = np.nan
        else:
            tpr_grupo = (y_predicho_grupo[positivos_reales] == 1).mean()

        tprs[g] = tpr_grupo

    diferencia = max(tprs.values()) - min(tprs.values())

    return tprs, diferencia


tprs_por_grupo, diferencia = calcular_diferencia_equidad(y_test, y_pred, grupo_test)

print("TPR por grupo:")
for g, tpr in tprs_por_grupo.items():
    print(f"  Grupo {g}: {tpr:.3f}")
print(f"Diferencia de oportunidad igual: {diferencia:.3f}")
```
</details>

## Ejercicio 2: estima qué variable pesa más en las decisiones del modelo

El modelo que auditaste en el ejercicio anterior decide a partir de tres variables: edad, ingresos y antigüedad laboral. Pero un modelo de aprendizaje automático no te dice por defecto qué variable pesa más en cada decisión: es una caja negra, como la consulta médica sin explicación que viste en la Clase 12. La importancia por permutación es una técnica sencilla de IA explicable (XAI) que abre esa caja: si permutas —barajas aleatoriamente— los valores de una columna y la precisión del modelo cae mucho, esa columna era importante para sus decisiones; si la precisión apenas cambia, el modelo casi no la usa.

Completa la función `importancia_por_permutacion(modelo, X, y, columna)`, que sigue estos pasos:

1. Calcula la precisión del modelo con los datos originales
2. Crea una copia de `X` y permuta —baraja aleatoriamente— solo los valores de la columna indicada, dejando las demás columnas intactas
3. Calcula la precisión del modelo con esa columna permutada
4. Devuelve la caída de precisión: precisión original menos precisión con la columna permutada

Reutiliza `modelo`, `X_test` y `y_test` del ejercicio 1.

**Pistas:**

- `accuracy_score(y_real, y_predicho)`, de `sklearn.metrics`, calcula la precisión
- `rng.permutation(X_permutado[:, columna])` devuelve una versión barajada de esa columna, sin modificar el resto de `X_permutado`
- Cuanto mayor es la caída de precisión, más importante es la variable para el modelo

In [ ]:
from sklearn.metrics import accuracy_score


def importancia_por_permutacion(modelo, X, y, columna, semilla=42):
    """
    Estima la importancia de una columna de X para las predicciones de
    `modelo`, midiendo cuánto empeora la precisión cuando se permutan
    (barajan aleatoriamente) sus valores.

    Parámetros
    ----------
    modelo : modelo ya entrenado, con método .predict()
    X : matriz de variables (numpy array)
    y : etiquetas reales
    columna : índice de la columna de X que quieres evaluar
    semilla : semilla del generador aleatorio, para que el resultado sea reproducible

    Devuelve
    --------
    caida_precision : precisión original menos precisión con la columna permutada
    """
    rng = np.random.default_rng(semilla)

    y_pred_original = modelo.predict(X)
    precision_original = accuracy_score(y, y_pred_original)

    X_permutado = X.copy()
    # TODO: sustituye la columna `columna` de X_permutado por una versión
    # barajada aleatoriamente de sus propios valores, usando rng.permutation
    X_permutado[:, columna] = ...

    y_pred_permutado = modelo.predict(X_permutado)
    precision_permutada = accuracy_score(y, y_pred_permutado)

    # TODO: calcula la caída de precisión (precisión original menos precisión permutada)
    caida_precision = ...

    return caida_precision


importancias = {}
for i, nombre in enumerate(nombres_columnas):
    importancias[nombre] = importancia_por_permutacion(modelo, X_test, y_test, i)

print("Caída de precisión al permutar cada variable:")
for nombre, caida in importancias.items():
    print(f"  {nombre}: {caida:.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(list(importancias.keys()), list(importancias.values()), color="#55A868", width=0.5)
ax.set_ylabel("Caída de precisión al permutar la variable")
ax.set_title("Importancia por permutación de cada variable")
plt.tight_layout()
plt.show()

<details>
<summary>💡 Ver solución</summary>

```python
from sklearn.metrics import accuracy_score


def importancia_por_permutacion(modelo, X, y, columna, semilla=42):
    """
    Estima la importancia de una columna de X para las predicciones de
    `modelo`, midiendo cuánto empeora la precisión cuando se permutan
    (barajan aleatoriamente) sus valores.
    """
    rng = np.random.default_rng(semilla)

    y_pred_original = modelo.predict(X)
    precision_original = accuracy_score(y, y_pred_original)

    X_permutado = X.copy()
    X_permutado[:, columna] = rng.permutation(X_permutado[:, columna])

    y_pred_permutado = modelo.predict(X_permutado)
    precision_permutada = accuracy_score(y, y_pred_permutado)

    caida_precision = precision_original - precision_permutada

    return caida_precision


importancias = {}
for i, nombre in enumerate(nombres_columnas):
    importancias[nombre] = importancia_por_permutacion(modelo, X_test, y_test, i)

print("Caída de precisión al permutar cada variable:")
for nombre, caida in importancias.items():
    print(f"  {nombre}: {caida:.3f}")
```
</details>

## Para seguir

Acabas de auditar un modelo por equidad y de abrir su caja negra con una técnica de interpretabilidad: dos piezas centrales de la IA responsable que viste en la Clase 12. Ese mismo patrón —medir antes de asumir, cuestionar antes de desplegar— separa un modelo técnicamente competente de un modelo que además es seguro para las personas a las que afecta.

Si quieres ir más allá, añade una tercera métrica de equidad, la paridad demográfica, y compárala con la diferencia de oportunidad igual sobre los mismos datos: ¿cuentan la misma historia o no?

**Aplica esta auditoría a un modelo real de tu equipo, aunque sea de forma informal.**